# VideoDB Understanding: Two-Pass Action Understanding

Analyze a video in 5-second intervals:

- **Pass 1 — evidence:** sample at 1 FPS and combine 5 frames into one 3-column sheet with ordinal labels, then describe what happens.
- **Pass 2 — actions:** sample at 2 FPS and split 10 frames into two 3-column sheets of 5 frames each, with ordinal and frame-ID labels. Send both sheets in one model request, using neighboring evidence and previous action state to build an action timeline.

You can read the output of either pass.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/preview/guides/indexing-v2/understanding/two-pass-action-understanding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install dependencies

In [ ]:
%pip install -q "videodb>=0.5.0" python-dotenv pandas yt-dlp

## 2. Connect to VideoDB

Set `VIDEO_DB_BASE_URL` in your environment or `.env` to the Runtime endpoint with two-pass support enabled.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")

conn = connect(api_key=api_key, base_url=os.environ["VIDEO_DB_BASE_URL"])
collection = conn.get_collection()

## 3. Choose a video

Download the sample video locally with `yt-dlp` and upload the file. Replace the URL with your own video, or reuse an uploaded video. Merging the video and audio streams needs `ffmpeg`, which Colab already has.

In [ ]:
!yt-dlp -q -f "bv*[ext=mp4][height<=1080]+ba[ext=m4a]/b[ext=mp4]/b" --merge-output-format mp4 -o laundry-folding-pov.mp4 "https://www.youtube.com/watch?v=wuRf25_zME0"

video = collection.upload(file_path="laundry-folding-pov.mp4")

# To reuse an uploaded video, replace the download and upload above with:
# video = collection.get_video("YOUR_VIDEO_ID")

video.play()

## 4. Run the two passes

`inputs` connects the passes. `context.inputs` selects neighboring evidence, while `context.state` carries action state between intervals.

Change `seconds`, `every`, `frames_per_sheet`, `columns`, or `labels` below to try different settings.

Both passes use 5 frames per sheet and 3 columns: three frames in the first row and two in the second. Each full 5-second interval produces one evidence sheet and two action sheets. A shorter final interval can produce fewer frames or sheets.

In [ ]:
MODEL = "google/gemini-3.8-flash"  # or "google/gemini-3.7-flash"

understanding = video.understand(
    segmentation={"type": "time", "seconds": 5},
    analyzers=[
        {
            "type": "vlm",
            "name": "evidence",
            "sampling": {"strategy": "interval", "every": 1},
            "preprocessing": [
                {
                    "type": "contact_sheet",
                    "frames_per_sheet": 5,
                    "columns": 3,
                    "labels": ["ordinal"],
                }
            ],
            "config": {
                "model": MODEL,
                "prompt": "Describe the visible actions and how people interact with objects.",
            },
        },
        {
            "type": "vlm",
            "name": "actions",
            "inputs": ["evidence"],
            "sampling": {"strategy": "interval", "every": 0.5},
            "preprocessing": [
                {
                    "type": "contact_sheet",
                    "frames_per_sheet": 5,
                    "columns": 3,
                    "labels": ["ordinal", "frame_id"],
                }
            ],
            "config": {
                "model": MODEL,
                "prompt": "Identify continuous actions and task stages. Keep an action open if it continues.",
                "context": {
                    "inputs": {"evidence": {"previous": 1, "current": True, "next": 1}},
                    "state": {
                        "policy": "action_timeline_v1",
                        "recent_segments": 3,
                        "recent_stages": 2,
                    },
                },
            },
        },
    ],
)

print("Understanding ID:", understanding.id)

In [ ]:
understanding.wait_until_complete(timeout=7200, poll_interval=15)
print("Status:", understanding.status)

## 5. View the results

Evidence contains descriptions for each interval. Actions can span several intervals.

In [ ]:
import pandas as pd

evidence_output = understanding.get_analyzer("evidence").get_output()
pd.json_normalize(evidence_output["scenes"]).head()

In [ ]:
actions_output = understanding.get_analyzer("actions").get_output()
pd.json_normalize(actions_output["scenes"]).head()

## 6. Robo activity: both passes in one analyzer

`robo_activity` runs a fixed two-pass pipeline for robotics-style action annotation. You choose only the model and the mode:

- **Pass 1** describes each 6-second window of frames and proposes action boundaries.
- **Pass 2** turns those descriptions into actions and goals, in pieces of up to 60 seconds.

`mode` defaults to `"batch"`, which runs both passes as Gemini batch jobs. Batch jobs cost less but can take from minutes to several hours.

In [ ]:
robo = video.understand(
    analyzers=[
        {
            "type": "robo_activity",
            "name": "robo",
            "config": {"model": MODEL, "mode": "batch"},
        }
    ],
)

print("Understanding ID:", robo.id)

In [ ]:
robo.wait_until_complete(timeout=24 * 3600, poll_interval=60)
print("Status:", robo.status)

Each scene is one action, with its instruction, goal and boundaries. The metadata lists the goals and a summary of the batch jobs.

In [ ]:
robo_output = robo.get_analyzer("robo").get_output()
pd.json_normalize(robo_output["scenes"]).head()

In [ ]:
pd.DataFrame(robo_output["metadata"]["goals"])